# AwesomeCUA catalog quality and specification coverage

**Grain:** one resource record, not one independent project. A paper, code repository, dataset release, and model checkpoint can be separate resources. Count observations keep their own units and populations.

**Purpose:** assess whether the catalog can support source-traceable research lookup and specification comparison. This checks catalog structure and evidence, not the quality or reproducibility of upstream experiments.

The baseline is source commit `5aad915`, before structured profiles were introduced. All checks below run locally from repository data without network calls. Install `requirements-dev.txt` if JSON Schema validation is unavailable.


In [1]:
from pathlib import Path
import collections, hashlib, json, subprocess, sys
import jsonschema
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data/catalog.jsonl").exists())
sys.path.insert(0, str(ROOT / "scripts"))
from build import validate_records
from specifications import PROFILE_KEYS, selected_profiles
path = ROOT / "data/catalog.jsonl"
rows = [json.loads(line) for line in path.read_text().splitlines()]
print("Catalog SHA-256:", hashlib.sha256(path.read_bytes()).hexdigest())
print("Source:", "data/catalog.jsonl")
print("Resource records:", len(rows))
print("Non-arXiv bibliography records:", sum(r["kind"] in {"paper", "survey"} and not r["id"].startswith("arxiv:") for r in rows))
print("Records with a publication edition:", sum(bool(r["details"].get("publication")) for r in rows))
archive_rows = [json.loads(line) for line in (ROOT / "data/papers.jsonl").read_text().splitlines()]
scope_status = json.loads((ROOT / "data/exclusion-status.json").read_text())
print("Retained arXiv metadata archive:", len(archive_rows))
print("Applied scope exclusions:", scope_status["excluded_records"])
print("Scope decisions requiring revalidation:", scope_status["revalidation_pending"])


Catalog SHA-256: 7f71a1683ab013e587a20086c3e357956981f517a24ae8bc8e81791108c66e4f
Source: data/catalog.jsonl
Resource records: 3048
Non-arXiv bibliography records: 13
Records with a publication edition: 77
Retained arXiv metadata archive: 2708
Applied scope exclusions: 122
Scope decisions requiring revalidation: 0


In [2]:
def profile(records):
    curated = [r for r in records if r["review_status"] == "curated"]
    result = {"resources": len(records), "curated": len(curated),
              "duplicate_resource_ids": len(records) - len({r["id"] for r in records}),
              "known_duplicate_bibliographic_records": sum(bool(r["identifiers"].get("preferred_arxiv")) for r in records),
              "evidence_retrieved_min": min(r["last_verified_at"] for r in records),
              "evidence_retrieved_max": max(r["last_verified_at"] for r in records)}
    result["by_kind"] = dict(sorted(collections.Counter(r["kind"] for r in curated).items()))
    result["structured_profile_coverage"] = {}
    for kind, key in PROFILE_KEYS.items():
        relevant = [r for r in curated if r["kind"] == kind]
        known = [r for r in relevant if r["details"].get(key)]
        result["structured_profile_coverage"][kind] = {"eligible_resources":len(relevant), "profiles":len(known),
            "missing_profiles":len(relevant)-len(known), "coverage_percent":round(100*len(known)/len(relevant),1) if relevant else None}
    return result
current = profile(rows)
try:
    text = subprocess.run(["git", "show", "5aad915:data/catalog.jsonl"], cwd=ROOT, text=True, capture_output=True, check=True).stdout
    baseline = profile([json.loads(line) for line in text.splitlines()])
    print("Baseline structured-profile coverage:", json.dumps(baseline["structured_profile_coverage"], indent=2))
except subprocess.CalledProcessError:
    baseline = None
    print("Baseline unavailable in this checkout; current-state checks still run.")
print("Current profile:", json.dumps(current, indent=2))


Baseline structured-profile coverage: {
  "benchmark": {
    "eligible_resources": 60,
    "profiles": 0,
    "missing_profiles": 60,
    "coverage_percent": 0.0
  },
  "dataset": {
    "eligible_resources": 41,
    "profiles": 0,
    "missing_profiles": 41,
    "coverage_percent": 0.0
  }
}
Current profile: {
  "resources": 3048,
  "curated": 1572,
  "duplicate_resource_ids": 0,
  "known_duplicate_bibliographic_records": 2,
  "evidence_retrieved_min": "2026-10-16T06:31:20Z",
  "evidence_retrieved_max": "2026-10-18T04:35:19Z",
  "by_kind": {
    "agent": 41,
    "benchmark": 83,
    "company": 16,
    "dataset": 85,
    "framework": 72,
    "infrastructure": 14,
    "model": 98,
    "paper": 1108,
    "product": 20,
    "survey": 15,
    "tool": 20
  },
  "structured_profile_coverage": {
    "benchmark": {
      "eligible_resources": 83,
      "profiles": 60,
      "missing_profiles": 23,
      "coverage_percent": 72.3
    },
    "dataset": {
      "eligible_resources": 85,
      "prof

## Checks and interpretation

The primary key is `id`; bibliography aliases are retained as provenance rather than silently merged. Dates are checked against source retrieval, and relationships must point to existing records. Retrieval time is not publication time and does not establish the freshness of every upstream source.

Specification coverage is measured only within curated benchmark/dataset records. Missing profiles are an editorial coverage gap, not proof of ingestion failure. The baseline had useful partial fields but no profiles in the new structured format.


In [3]:
schema = json.loads((ROOT / "schemas/resource.schema.json").read_text())
validator = jsonschema.Draft202012Validator(schema, format_checker=jsonschema.FormatChecker())
schema_errors = [(r["id"], e.message) for r in rows for e in validator.iter_errors(r)]
semantic_errors = validate_records(rows)
from scope_review import resolve_exclusions
retained = {row["id"]: row for row in archive_rows}
retained.update({row["id"]: row for row in rows})
proposals = [json.loads(line) for line in (ROOT / "data/metadata-updates.jsonl").read_text().splitlines()]
outcomes = resolve_exclusions(retained, json.loads((ROOT / "data/exclusions.json").read_text()), {p["id"] for p in proposals})
excluded_ids = {d["id"] for d in outcomes if d["status"] == "excluded"}
assert not excluded_ids.intersection({r["id"] for r in rows})
assert excluded_ids.issubset({r["id"] for r in archive_rows})
print("Scope decisions preserve raw metadata and do not leak into the active catalog:", len(excluded_ids))
print("JSON Schema errors:", len(schema_errors))
print("Identity, date, relation, evidence, and count-grain errors:", len(semantic_errors))
assert not schema_errors, schema_errors[:5]
assert not semantic_errors, semantic_errors[:5]


Scope decisions preserve raw metadata and do not leak into the active catalog: 122
JSON Schema errors: 0
Identity, date, relation, evidence, and count-grain errors: 0


In [4]:
for kind, key in PROFILE_KEYS.items():
    profiles = selected_profiles(rows, kind)
    # This counts evidence observations by unit; it never sums their numeric values.
    observations = [c for r in profiles for c in r["details"][key]["counts"]]
    frequencies = collections.Counter(c["unit"] for c in observations)
    print(kind, "profile rows:", len(profiles), "count observations:", len(observations))
    print("Number of count observations by unit:", dict(sorted(frequencies.items())))
    print("Profiles without a count:", sum(not r["details"][key]["counts"] for r in profiles))


benchmark profile rows: 60 count observations: 110
Number of count observations by unit: {'Android applications': 1, 'VM task definitions': 1, 'Windows applications': 1, 'annotation rows': 2, 'application categories': 1, 'application families': 1, 'applications': 8, 'category/task references': 1, 'cross-app tasks': 1, 'dataset families': 1, 'desktop applications': 1, 'environment configuration rows': 2, 'examples': 1, 'expert demonstrations': 1, 'federated learning algorithms': 1, 'heterogeneity types': 1, 'human demonstrations': 2, 'human trajectories': 1, 'instruction types': 1, 'instructions': 1, 'instruction–target examples': 1, 'interactions': 1, 'mobile applications': 2, 'mobile apps': 1, 'page-annotation entries': 2, 'platform JSONL splits': 1, 'products': 1, 'rows': 1, 'simulated apps': 1, 'simulated clients': 2, 'single-app tasks': 1, 'target annotations': 1, 'task IDs': 4, 'task definitions': 8, 'task examples': 4, 'task identifiers': 2, 'task identities': 1, 'task images': 1

## Findings, risks, and remediation

- **Medium severity; high confidence — incomplete comparison coverage.** The coverage output quantifies profiles present and missing within the current catalog. Use the comparison pages as a documented subset, and continue editorial review of uncovered assets. Do not interpret absent fields as zero or as missing capabilities.
- **High potential impact; high confidence — incompatible counting units.** The unit inventory includes task definitions, instances, episodes, screens, annotations, and actions. Combining their values would misstate scale. Schema and semantic checks require units, populations, qualifiers, and evidence URLs and reject duplicate count grain.
- **Medium severity; high confidence — conflicting GUIOdyssey prose.** The project README contains both 8,334 and 8,834 episodes. The specifically recommended release card states 8,334; that version-specific value is retained, together with the conflict note. It is a publisher declaration, not an independent archive recount.
- **Medium severity; high confidence — split and annotation limitations matter.** WorkDemo-350 reports post hoc subset selection and unresolved coordinate/acceptance limitations. MoTIF mixes trace types with different annotation coverage. These source-specific limitations remain visible in their records and profiles.
- **Explicit scope filtering; high confidence.** Individually reviewed exclusions can reduce the active catalog row count while preserving the raw archive. This is an intentional scope decision, not data loss. The reviewed set is targeted, so its exclusion rate does not estimate overall discovery precision. Changed or unconfirmed decisions reopen review instead of silently suppressing records.
- **No independent experimental or legal assurance.** Passing catalog checks does not validate model performance, dataset annotation quality, or permissions beyond the publisher's declared terms.

There is no stable temporal baseline for the new profile fields beyond the introduction comparison. No trend or anomaly threshold is inferred from two snapshots.

## Sources and reproducibility

- Current records: `data/catalog.jsonl`; raw inputs: `data/editorial.json`, `data/primary-evidence.jsonl`, `data/enrichment.jsonl`, `data/papers.jsonl`, and `data/publications.jsonl`.
- Baseline: https://github.com/reacher-z/Awesome-CUA/tree/5aad915
- Contract: `schemas/resource.schema.json` and `docs/specifications.md`.
- Checks: `scripts/build.py`, `scripts/specifications.py`, and the tests under `tests/`.
- Every exported profile includes original source URLs, retrieval timestamps, and content hashes; count observations additionally link their own evidence. The scope and unit remain necessary when reusing a quantity.
